<a href="https://colab.research.google.com/github/Yusep-T/data-science-2026/blob/main/Pertemuan3_Yusep_250401020103.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Pertemuan 3 - Data Cleaning Housing Dataset

Notebook ini berisi proses data cleaning sesuai instruksi tugas.

In [3]:
# 1. Import library
import pandas as pd
import numpy as np
from scipy.stats.mstats import winsorize


In [4]:
# 2. Muat dataset
# Jika menggunakan Google Colab, upload file housing_dirty.csv terlebih dahulu
df = pd.read_csv('/content/sample_data/housing_dirty.csv')

# Tampilkan 5 data pertama
df.head()


,id,luas_m2,harga_juta,kota,kamar,tahun_bangun,kondisi
0,1,297.0,1084.0,jogja,2.0,2000,baik
1,2,254.0,761.0,Medan,NaN,1995,Bagus
2,3,249.7,895.0,Depok,NaN,1983,baik
3,4,49.7,178.0,YGY,5.0,2013,baik
4,5,133.4,424.0,Medan,5.0,2004,Sedang


In [5]:
# 3. Eksplorasi awal
print("Info Dataset")
print(df.info())

print("\nDeskripsi Statistik")
display(df.describe())

print("\nJumlah Missing Values")
display(df.isnull().sum())


Info Dataset
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 130 entries, 0 to 129
Data columns (total 7 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   id            130 non-null    int64  
 1   luas_m2       112 non-null    float64
 2   harga_juta    113 non-null    float64
 3   kota          130 non-null    object 
 4   kamar         120 non-null    float64
 5   tahun_bangun  130 non-null    int64  
 6   kondisi       130 non-null    object 
dtypes: float64(3), int64(2), object(2)
memory usage: 7.2+ KB
None

Deskripsi Statistik


,id,luas_m2,harga_juta,kamar,tahun_bangun
count,130.000000,112.000000,1.130000e+02,120.000000,130.000000
mean,65.500000,267.627679,8.856325e+05,3.433333,2062.638462
std,37.671829,885.664181,9.407144e+06,1.776283,701.684043
min,1.000000,-50.000000,-5.000000e+02,1.000000,1890.000000
25%,33.250000,87.050000,3.450000e+02,2.000000,1991.250000
50%,65.500000,193.800000,6.550000e+02,4.000000,2002.000000
75%,97.750000,280.675000,9.550000e+02,5.000000,2011.750000
max,130.000000,9500.000000,1.000000e+08,6.000000,9999.000000



Jumlah Missing Values


,0
id,0
luas_m2,18
harga_juta,17
kota,0
kamar,10
tahun_bangun,0
kondisi,0


In [6]:
# 4. Hapus baris duplikat
df.drop_duplicates(inplace=True)

print("Jumlah data duplikat setelah dibersihkan:", df.duplicated().sum())


Jumlah data duplikat setelah dibersihkan: 0


In [7]:
# 5. Normalisasi string

# Kolom kota
df['kota'] = df['kota'].astype(str).str.strip().str.title()

# Kolom kondisi
df['kondisi'] = df['kondisi'].astype(str).str.strip().str.lower()

# Cek hasil
df[['kota', 'kondisi']].head()


,kota,kondisi
0,Jogja,baik
1,Medan,bagus
2,Depok,baik
3,Ygy,baik
4,Medan,sedang


In [8]:
# 6. Imputasi missing values

# Kolom numerik
numeric_cols = df.select_dtypes(include=['int64', 'float64']).columns

for col in numeric_cols:
    median_value = df[col].median()
    df[col] = df[col].fillna(median_value)

# Kolom kategorik
categorical_cols = df.select_dtypes(include=['object']).columns

for col in categorical_cols:
    mode_value = df[col].mode()[0]
    df[col] = df[col].fillna(mode_value)

# Cek missing values
df.isnull().sum()


,0
id,0
luas_m2,0
harga_juta,0
kota,0
kamar,0
tahun_bangun,0
kondisi,0


In [9]:
# 7. Menangani outlier dengan metode IQR Fence

def remove_outlier_iqr(dataframe, column):
    Q1 = dataframe[column].quantile(0.25)
    Q3 = dataframe[column].quantile(0.75)
    IQR = Q3 - Q1

    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    dataframe = dataframe[
        (dataframe[column] >= lower_bound) &
        (dataframe[column] <= upper_bound)
    ]

    return dataframe

# Terapkan pada kolom harga_juta dan luas_m2
df = remove_outlier_iqr(df, 'harga_juta')
df = remove_outlier_iqr(df, 'luas_m2')

print("Jumlah data setelah menghapus outlier:", len(df))


Jumlah data setelah menghapus outlier: 126


In [10]:
# 8. Validasi akhir

print("Total missing values:", df.isnull().sum().sum())
print("Total duplicated rows:", df.duplicated().sum())

assert df.isnull().sum().sum() == 0
assert df.duplicated().sum() == 0

print("Validasi berhasil!")


Total missing values: 0
Total duplicated rows: 0
Validasi berhasil!


In [11]:
# 9. Ekspor dataset bersih
df.to_csv('housing_clean.csv', index=False)

print("File housing_clean.csv berhasil disimpan")


File housing_clean.csv berhasil disimpan


In [12]:
# 10. Akses API JSONPlaceholder dan simpan ke DataFrame
api_url = 'https://jsonplaceholder.typicode.com/posts'

api_df = pd.read_json(api_url)

# Tampilkan data API
api_df.head()


,userId,id,title,body
0,1,1,sunt aut facere repellat provident occaecati e...,quia et suscipit\nsuscipit recusandae consequu...
1,1,2,qui est esse,est rerum tempore vitae\nsequi sint nihil repr...
2,1,3,ea molestias quasi exercitationem repellat qui...,et iusto sed quo iure\nvoluptatem occaecati om...
3,1,4,eum et est occaecati,ullam et saepe reiciendis voluptatem adipisci\...
4,1,5,nesciunt quas odio,repudiandae veniam quaerat sunt sed\nalias aut...


In [13]:
# 11. Simpan DataFrame API jika diperlukan
api_df.to_csv('jsonplaceholder_posts.csv', index=False)

print("Data API berhasil disimpan")


Data API berhasil disimpan
